# Example 7.10 / 1: Strengthening an Activation Model

Companion notebook for **7.10 Formulation Strength** in the AMPL lab Documentation, section "Strengthening an Activation Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Strengthening an Activation Model

Suppose `0 <= x[i] <= U[i]` and binary `y` activates the entire group at a fixed cost. The
aggregate link `TotalActivation` has the same integer meaning as the disaggregated links
`Activation`: when `y = 0`, every nonnegative `x[i]` is zero, and when `y = 1`, the original
upper bounds apply. The model below contains both, so each can be dropped in turn. Only `x[1]`
is worth anything, 5 per unit, and switching the group on costs 15:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set I;                            # members of the group
param U {I} > 0;                  # upper bound of each member
param gain {I};                   # value per unit of each member
param fixed_cost >= 0;            # cost of switching the group on
var x {i in I} >= 0, <= U[i];     # activity of each member
var y binary;                     # 1 if the group is switched on

maximize Net: sum {i in I} gain[i] * x[i] - fixed_cost * y;
# aggregate link: one row for the whole group, weaker when y is fractional
subject to TotalActivation: sum {i in I} x[i] <= (sum {i in I} U[i]) * y;
# disaggregated links: one row per member, stronger relaxation
subject to Activation {i in I}: x[i] <= U[i] * y;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["I"] = [1, 2, 3]
ampl.param["U"] = {1: 4, 2: 6, 3: 10}
ampl.param["gain"] = {1: 5, 2: 0, 3: 0}
ampl.param["fixed_cost"] = 15

In [5]:
# run.py
for keep, drop in [("TotalActivation", "Activation"), ("Activation", "TotalActivation")]:
    ampl.con[drop].drop()
    for relax in (1, 0):
        ampl.option["relax_integrality"] = relax
        ampl.solve(solver="highs", verbose=False)
        print(f"{keep:16} {'relaxation' if relax else 'integer':10} Net = {ampl.obj['Net'].value():g}",
              f" y = {ampl.var['y'].value():g}")
    ampl.con[drop].restore()

TotalActivation  relaxation Net = 17  y = 0.2
TotalActivation  integer    Net = 5  y = 1
Activation       relaxation Net = 5  y = 1
Activation       integer    Net = 5  y = 1


### Pure AMPL: data.dat and model.run

Both formulations reach the same integer optimum, 5. The aggregate relaxation sets `y = 0.2`,
enough for `x[1] = 4` because the row counts the room of all three members, and claims a bound
of 17. The disaggregated rows limit `x[1]` to `4 * y` on its own, so the relaxation already
equals the integer optimum. The individual rows are stronger when `y` is fractional: capacity
available to one member cannot justify another member exceeding `U[i] * y`. This conclusion
depends on the stated nonnegativity and upper bounds; equivalence must always be proved from the
complete model.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: I: U gain := 1 4 5  2 6 0  3 10 0;
param fixed_cost := 15;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
option relax_integrality 1;
drop Activation;
solve;
printf "TotalActivation relaxation Net = %g\n", Net;
restore Activation;
drop TotalActivation;
solve;
printf "Activation relaxation Net = %g\n", Net;

TotalActivation relaxation Net = 17
Activation relaxation Net = 5
